# 13 FS4 Huang-Style Feature And Target Analysis

This notebook is a **methodology-first skeleton** for the future `FS4` workstream on similar-days features.

Its job is to justify the **pre-retrieval design choices** before any FS4 feature construction begins:
- why similar-days should be a separate feature layer rather than an invisible `FS3` extension
- which candidate exogenous and historical families are causally admissible at the forecast origin
- how daily local-hour profiles should be represented in the Dutch hourly day-ahead context
- how Huang's descriptive-analysis flow should be adapted to the thesis scope

This notebook must stay descriptive and design-oriented. It must **not** run FS4 model training or create the final FS4 feature store.


In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root
fs4_methodology_doc = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices/docs/fs4_huang_style_methodology_plan.md"

print(fs4_methodology_doc)
print(config.to_json_dict())


## 1. Notebook Intent

This notebook should later answer four preparatory questions.

1. What is the conceptual role of `FS4` in the thesis, and why is it not the same as ordinary `FS3`?
2. Which candidate feature families are actually **causally known** at `D-1 08:00` local in the current repo?
3. How should a Dutch day-ahead delivery day be represented as a 24-slot local-day profile while UTC storage remains intact?
4. What descriptive evidence should be reported before similar-day retrieval rules are frozen?

The written reference for those decisions is:
- `scripts/Data/02_Forecasting/01_DA_prices/docs/fs4_huang_style_methodology_plan.md`


## 2. FS4 As A Separate Feature Layer

`FS4` should be framed as a **retrieval-based feature engineering layer**.

It differs from `FS3` because it does not merely pass direct exogenous variables into the model. Instead it:
- defines a target-day representation
- retrieves analogous historical days from a causal candidate pool
- transforms those retrieved days into new price-profile features

This notebook should later make that distinction explicit so the thesis can defend why `FS4` remains a separate experiment dimension.

TODO for later execution:
- write a short workflow figure or structured flow summary
- state the exact causal rule `known_at_utc <= forecast_origin_utc`
- explain why the benchmark stack stays frozen before any FS4 model comparison begins


## 3. Huang-Inspired Flow Adapted To Hourly NL DA Forecasting

Huang's flow should be mirrored in this order:
1. within-day descriptive statistics of target and candidate exogenous drivers
2. Pearson-correlation screening between candidate drivers and the target
3. daily price-pattern clustering
4. only after that: similar-day retrieval analysis

Adaptation to this thesis:
- Huang works on `96` quarter-hour points; here the analytical object is a `24` local-hour profile
- the active market is `NL`
- the business horizon is `D..D+4`, but similar-day design should begin as a **`D`-only local-day profile problem**
- internal storage and leakage checks stay in UTC even when the analytical profile is in local market-day space

TODO for later execution:
- add an explicit note on DST profile harmonization
- add a note that test data must stay untouched while the FS4 methodology is still being frozen


## 4. Candidate Feature-Family Causality Review

The future executed version of this notebook should review candidate families in four groups:
- raw exogenous profiles known at the origin
- derived exogenous profiles such as residual-style proxies
- historical price profiles
- lagged historical profile families

First-pass families to review positively:
- `NL` day-ahead total load forecast profile for `D`
- `NL` day-ahead total generation forecast profile for `D`
- `NL` week-ahead total load forecast profile as the guidance-safe bridge family
- historical `NL` DA price profiles

Families that must be treated cautiously:
- residual or net-load proxies derived from the day-ahead load and generation profiles
- cross-border proxy families
- installed-capacity profiles, which are causally safe but weak as intraday shape drivers

Families that should **not** be assumed present just because Huang uses analogous ideas:
- future-known wind forecast profiles
- future-known solar / PV forecast profiles

TODO for later execution:
- build a causality table with columns such as `family`, `scope`, `known_at_rule`, `target-day usability`, `expected relevance`, `first-pass decision`
- document explicitly which families are unavailable in the current repo and therefore excluded from the first pass


## 5. Within-Day Profile Visualization Design

Huang-style reporting starts with descriptive plots. In this repo, those plots should be designed around **local-hour profiles** rather than raw UTC sequences.

Planned outputs:
- a multi-panel figure showing within-day distributions for the target price and each retained candidate driver
- a clear note on profile harmonization across DST
- profile-quality diagnostics showing observed-hour coverage and imputation share

Methodological note:
- because the cleaned target series contains real source gaps in addition to DST anomalies, profile quality cannot be assumed from the calendar alone
- gap-filled target values may be used for feature construction only, never for scoring

TODO for later execution:
- define the profile-quality thresholds to be used in the descriptive plots
- decide whether heavily incomplete non-DST days are excluded or shown separately in a data-quality appendix cell


## 6. Descriptive Statistics And Pearson-Correlation Reporting

This section should later produce a Huang-style table with at least:
- mean
- standard deviation
- min / max
- daily range
- Pearson correlation with the target price

Recommended adaptation:
- compute the descriptive table on train plus validation only while the FS4 methodology is still being frozen
- report correlations transparently as descriptive evidence rather than as final proof of feature usefulness
- keep the later feature-selection phase separate from this preparatory notebook

TODO for later execution:
- define whether correlations are computed from hourly pooled observations, daily profile summaries, or both
- explain any difference between price-shape correlation and daily-level correlation


## 7. Daily Price-Pattern Clustering Design

This notebook should also prepare the Huang-style daily price-pattern analysis.

Recommended first method:
- cluster harmonized `24`-slot local-day `NL` price profiles with `k`-means
- test a small range such as `k = 3..8`
- choose `k` using quantitative diagnostics plus interpretability

Planned reporting outputs:
- centroid-profile figure
- cluster prevalence table
- cluster summary table with price level, volatility, and seasonal composition

Why this belongs here:
- cluster structure is part of the argument that analogous daily regimes exist at all
- later cluster-conditioned similar-day features depend on this preparatory evidence

TODO for later execution:
- define the exact train versus validation role in cluster fitting and checking
- state clearly that the future target day's unknown price cluster cannot be used directly at forecast time


## 8. Design Decisions To Freeze Before Notebook 14

Before moving into similar-day retrieval analysis, the following choices should be frozen here:
- the retained candidate feature families
- the causal admissibility rule for each family
- the local-day profile harmonization rule, including DST handling
- the profile-quality filter
- the clustering protocol and cluster-reporting structure

Notebook `14` should start only after those design choices are written down explicitly.


In [ ]:
FS4_NOTEBOOK_13_FREEZE_CHECKLIST = {
    "fs4_role_frozen": False,
    "candidate_family_list_frozen": False,
    "causal_availability_rules_frozen": False,
    "local_day_profile_rule_frozen": False,
    "dst_harmonization_rule_frozen": False,
    "profile_quality_filter_frozen": False,
    "price_clustering_protocol_frozen": False,
}

FS4_NOTEBOOK_13_TODOS = [
    "Load the retained candidate families from the cleaned feature inventory.",
    "Build descriptive within-day distribution figures in local-hour space.",
    "Produce the descriptive-statistics-plus-correlation table.",
    "Run and document the price-profile clustering study.",
    "Freeze the design decisions that notebook 14 depends on.",
]

pd.DataFrame(
    {
        "item": list(FS4_NOTEBOOK_13_FREEZE_CHECKLIST.keys()),
        "frozen": list(FS4_NOTEBOOK_13_FREEZE_CHECKLIST.values()),
    }
)
